In [3]:
import cv2
import torch
import numpy as np
from ultralytics import YOLO
import torchvision.transforms as transforms
from transformers import Swin2SRImageProcessor, Swin2SRForImageSuperResolution


# 1. LOAD MODELS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

yolo = YOLO("runs/detect/train7/weights/best.pt")

base_model_name = "caidas/swin2SR-realworld-sr-x4-64-bsrgan-psnr"
sr_processor = Swin2SRImageProcessor.from_pretrained(base_model_name)
sr_model = Swin2SRForImageSuperResolution.from_pretrained(base_model_name).to(device)
sr_model.load_state_dict(torch.load("swin2sr_v2_plate_epoch2.pth", map_location=device))
sr_model.eval()

class OCR_CNN_Deep(torch.nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, 3, padding=1), torch.nn.BatchNorm2d(32), torch.nn.ReLU(),
            torch.nn.Conv2d(32, 32, 3, padding=1), torch.nn.BatchNorm2d(32), torch.nn.ReLU(),
            torch.nn.MaxPool2d(2), torch.nn.Dropout(0.25),
            torch.nn.Conv2d(32, 64, 3, padding=1), torch.nn.BatchNorm2d(64), torch.nn.ReLU(),
            torch.nn.Conv2d(64, 64, 3, padding=1), torch.nn.BatchNorm2d(64), torch.nn.ReLU(),
            torch.nn.MaxPool2d(2), torch.nn.Dropout(0.25),
            torch.nn.Conv2d(64, 128, 3, padding=1), torch.nn.BatchNorm2d(128), torch.nn.ReLU(),
            torch.nn.Conv2d(128, 128, 3, padding=1), torch.nn.BatchNorm2d(128), torch.nn.ReLU(),
            torch.nn.MaxPool2d(2), torch.nn.Dropout(0.25),
            torch.nn.Conv2d(128, 256, 3, padding=1), torch.nn.BatchNorm2d(256), torch.nn.ReLU(),
            torch.nn.Conv2d(256, 256, 3, padding=1), torch.nn.BatchNorm2d(256), torch.nn.ReLU(),
            torch.nn.MaxPool2d(2), torch.nn.Dropout(0.25),
        )
        self.classifier = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(256 * 2 * 2, 512), torch.nn.ReLU(), torch.nn.Dropout(0.5),
            torch.nn.Linear(512, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

ocr_model = OCR_CNN_Deep(num_classes=36).to(device)
ocr_model.load_state_dict(torch.load("ocr_model_deep_v1.pth", map_location=device))
ocr_model.eval()

class_names = list("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ")
AMBIGUOUS = {"0": "O", "O": "0", "1": "I", "I": "1", "8": "B", "B": "8"}

transform = transforms.Compose([
    transforms.ToPILImage(), transforms.Grayscale(),
    transforms.Resize((32, 32)), transforms.ToTensor()
])


# 2. UTILITY FUNCTIONS & PIPELINE

def is_red_background(char_rgb):
    hsv = cv2.cvtColor(char_rgb, cv2.COLOR_RGB2HSV)
    mask1, mask2 = cv2.inRange(hsv, (0, 70, 50), (10, 255, 255)), cv2.inRange(hsv, (170, 70, 50), (180, 255, 255))
    return (cv2.bitwise_or(mask1, mask2).mean() / 255.0) > 0.15

def preprocess_dark_on_dark(char_rgb):
    return cv2.normalize(cv2.cvtColor(char_rgb, cv2.COLOR_RGB2GRAY), None, 0, 255, cv2.NORM_MINMAX)

def is_dark_on_dark(char_rgb):
    gray = cv2.cvtColor(char_rgb, cv2.COLOR_RGB2GRAY)
    return gray.mean() < 80 and (gray.max() - gray.min()) < 60

def upscale_with_swin(img_rgb):
    h_orig, w_orig = img_rgb.shape[:2]
    with torch.no_grad():
        outputs = sr_model(**sr_processor(img_rgb, return_tensors="pt").to(device))
    output_img = np.moveaxis(outputs.reconstruction.data.cpu().float().clamp(0, 1).numpy()[0], 0, -1)
    return (output_img[:h_orig * 4, :w_orig * 4, :] * 255.0).round().astype(np.uint8)

def predict_character(char_img_rgb):
    if is_red_background(char_img_rgb): char_img_rgb = cv2.bitwise_not(cv2.cvtColor(char_img_rgb, cv2.COLOR_RGB2GRAY))
    elif is_dark_on_dark(char_img_rgb): char_img_rgb = preprocess_dark_on_dark(char_img_rgb)
    
    with torch.no_grad():
        conf, pred = torch.softmax(ocr_model(transform(char_img_rgb).unsqueeze(0).to(device)), dim=1).max(dim=1)
    return class_names[pred.item()], conf.item()

def disambiguate_plate(chars):
    chars = chars.copy()
    for i, ch in enumerate(chars):
        if ch in AMBIGUOUS: chars[i] = AMBIGUOUS[ch] if (i < 3 and ch.isdigit()) or (i >= 3 and ch.isalpha()) else chars[i]
    return chars

def sort_boxes_reading_order(boxes, confs):
    chars = [{"box": (x1, y1, x2, y2), "conf": c, "cx": (x1+x2)/2, "cy": (y1+y2)/2, "h": y2-y1} 
             for (x1, y1, x2, y2), c in zip(boxes, confs)]
    avg_h, rows = np.mean([c["h"] for c in chars]), []
    for c in chars:
        if not any(row.append(c) or True for row in rows if abs(c["cy"] - row[0]["cy"]) < avg_h * 0.6): rows.append([c])
    rows.sort(key=lambda r: np.mean([c["cy"] for c in r]))
    for row in rows: row.sort(key=lambda c: c["cx"])
    return [c["box"] for row in rows for c in row], [c["conf"] for row in rows for c in row]

def filter_false_positives(boxes, confs, img_h):
    if len(boxes) == 0: return [], []
    valid = [(i, y2-y1) for i, (_, y1, _, y2) in enumerate(boxes) if (y2-y1) >= img_h * 0.10]
    if not valid: return [], []
    median_h = np.median([h for _, h in valid])
    return zip(*[(boxes[i], confs[i]) for i, h in valid if 0.5 * median_h < h < 1.5 * median_h]) or ([], [])

def recognize_plate(image_path, conf=0.35):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None: return "IMAGE_NOT_FOUND"
    
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h_orig, w_orig = img_rgb.shape[:2]

    results = yolo(img_bgr, conf=conf, iou=0.3, verbose=False)
    if len(results[0].boxes) == 0: return "NO_DETECTION"

    boxes, confs = filter_false_positives(results[0].boxes.xyxy.cpu().numpy(), results[0].boxes.conf.cpu().numpy(), h_orig)
    if len(boxes) == 0: return "NO_DETECTION"

    boxes, confs = sort_boxes_reading_order(boxes, confs)
    chars = []

    for (x1, y1, x2, y2), _ in zip(boxes, confs):
        margin_x, margin_y = 0.20 * (x2 - x1), 0.20 * (y2 - y1)
        crop_orig = img_rgb[int(max(0, y1 - margin_y)):int(min(h_orig, y2 + margin_y)), 
                            int(max(0, x1 - margin_x)):int(min(w_orig, x2 + margin_x))]
        if crop_orig.size == 0: continue

        try: crop_sr = upscale_with_swin(crop_orig)
        except Exception: crop_sr = crop_orig

        h_sr, w_sr = crop_sr.shape[:2]
        trim_x, trim_y = int(0.10 * w_sr), int(0.10 * h_sr)
        crop_sr_tight = crop_sr[trim_y:h_sr - trim_y, trim_x:w_sr - trim_x]
        
        ch, _ = predict_character(crop_sr_tight if crop_sr_tight.size > 0 else crop_sr)
        chars.append(ch)

    return "".join(disambiguate_plate(chars))


# 3. SINGLE IMAGE EXECUTION

if __name__ == "__main__":
    test_image_path = "C:/Users/Stanly/ALPR/Compressed/yj4Iu2-UFPR-ALPR/ALPR_OCR/train/images/track0060[01].png" # Replace with your image
    
    recognized_text = recognize_plate(test_image_path, conf=0.35)
    print(f"Recognized Plate: {recognized_text}")

Recognized Plate: APB4327
